# Zomato Restaurant Rating Predictor
### Data Preprocessing & Feature Engineering

This notebook covers the full preprocessing pipeline for the Zomato Bangalore dataset:
- Exploratory Data Analysis (EDA)
- Data Cleaning
- Feature Engineering (binary encoding, target encoding)
- Saving cleaned data and artifacts for model training

## 1. Imports & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

In [ ]:
df = pd.read_csv('Zomato.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Target variable distribution
df['rate (out of 5)'].describe()

In [ ]:
# Restaurant type distribution
df['restaurant type'].value_counts()

In [ ]:
# Area distribution
df['area'].value_counts()

## 3. Data Cleaning

Key steps:
- Rename columns for consistency
- Drop irrelevant columns (`Unnamed: 0`, `Unnamed: 9`)
- Clean the `area` column — some rows had multi-part entries like `Byresandra,Tavarekere,Madiwala`. We extract the last part as the canonical area name.

In [ ]:
df = df.rename(columns={
    'restaurant name': 'name',
    'restaurant type': 'rest_type',
    'rate (out of 5)': 'rate',
    'num of ratings': 'num_ratings',
    'avg cost (two people)': 'avg_cost',
    'table booking': 'table_booking',
    'cuisines type': 'cuisines',
    'local address': 'local_address'
})
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 9'], errors='ignore')
print(df.columns.tolist())

In [ ]:
# Extract canonical area name from multi-part entries
df['area_clean'] = df['area'].str.split(',').str[-1].str.strip()

print(f'Unique areas reduced from {df["area"].nunique()} to {df["area_clean"].nunique()}')
print('\nExample fix:')
print(df[df['area'].str.contains('Byresandra')][['area', 'area_clean']].head(3))

In [ ]:
# Encode online_order and table_booking as binary (Yes=1, No=0)
df['online_order'] = df['online_order'].map({'Yes': 1, 'No': 0})
df['table_booking'] = df['table_booking'].map({'Yes': 1, 'No': 0})

## 4. Feature Engineering

### 4a. Restaurant Type — Binary Encoding
Each restaurant type becomes a binary column (`is_quick_bites`, `is_cafe`, etc.).
We use `str.contains()` because a restaurant can belong to multiple types.

### 4b. Cuisine — Binary Encoding
Same approach for cuisines. Each cuisine gets its own binary column.

### 4c. Area — Target Encoding
Instead of one-hot encoding area (which would create 50+ columns), we replace each area with the **mean rating of all restaurants in that area**. This captures the area's quality signal in a single numeric column (`encoded_area`).

### 4d. Additional Features
- `area_avg_cost` — average cost per area (captures price tier of the neighborhood)
- `competition_density` — number of same-type restaurants in the same area
- `price_position` — restaurant's cost relative to its area average (above/below average price)

In [ ]:
# Binary encoding for restaurant types
main_types = [
    'Quick Bites', 'Casual Dining', 'Cafe',
    'Delivery', 'Dessert Parlor', 'Bakery',
    'Beverage Shop', 'Bar', 'Food Court',
    'Fine Dining', 'Lounge', 'Sweet Shop', 'Pub'
]
for t in main_types:
    col_name = 'is_' + t.lower().replace(' ', '_')
    df[col_name] = df['rest_type'].str.contains(t).astype(int)

# Verify encoding
binary_cols = ['is_' + t.lower().replace(' ', '_') for t in main_types]
print(df[binary_cols].sum())

In [ ]:
# Binary encoding for cuisines
all_cuisine = df['cuisines'].str.split(',').explode().str.strip().unique()

for c in all_cuisine:
    col_name = 'is_' + c.lower().replace(' ', '_')
    df[col_name] = df['cuisines'].str.contains(c).astype(int)

In [ ]:
# Area-level aggregate features
df['area_avg_rating'] = df.groupby('area_clean')['rate'].transform('mean')
df['area_avg_cost'] = df.groupby('area_clean')['avg_cost'].transform('mean')
df['competition_density'] = df.groupby(['area_clean', 'rest_type'])['name'].transform('count')
df['price_position'] = df['avg_cost'] / df['area_avg_cost']

In [ ]:
# Target encoding for area: replace area name with mean rating of that area
area_rating = df.groupby('area_clean')['rate'].mean()
df['encoded_area'] = df['area_clean'].map(dict(area_rating))

# Save area_rating mapping for use in Streamlit app
pickle.dump(area_rating, open('area_rating.pkl', 'wb'))
print('area_rating.pkl saved')

## 5. Final Cleanup

- Drop non-numeric / text columns that have been encoded
- Drop low-frequency cuisine columns (< 50 occurrences) to reduce noise
- Save final feature set

In [ ]:
# Drop original text columns — all information is now encoded
df = df.drop(columns=['name', 'rest_type', 'cuisines', 'area', 'local_address', 'area_clean'], errors='ignore')
print(f'Shape after dropping text columns: {df.shape}')

In [ ]:
# Drop low-frequency cuisine binary columns (< 50 restaurants)
# These add noise without enough signal for the model to learn from
is_cols = df.columns[df.columns.str.startswith('is_')]
is_sum = df[is_cols].sum()
low_freq = is_sum[is_sum < 50].index

print(f'Dropping {len(low_freq)} low-frequency columns: {list(low_freq)}')
df = df.drop(columns=low_freq, errors='ignore')
print(f'Final shape: {df.shape}')

In [ ]:
df.head()

## 6. Save Artifacts

In [ ]:
# Save cleaned dataset
df.to_csv('zomato_cleaned.csv', index=False)
print('zomato_cleaned.csv saved')

# Save feature column names (used by Streamlit app to build input DataFrame)
pickle.dump(df.drop(columns=['rate']).columns, open('feature_columns.pkl', 'wb'))
print('feature_columns.pkl saved')

# Verify
feature_columns = pickle.load(open('feature_columns.pkl', 'rb'))
print(f'Total features: {len(feature_columns)}')
print(f'Feature columns: {list(feature_columns)}')